<a href="https://colab.research.google.com/github/Zekeriya-Ui/main/blob/main/Flewd_Search_Intelligence_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Resources
Here is your complete, production-ready submission for **Assignment FL-06: Design Your Personal Agent**, tailored specifically for your FlyRank AI Hackathon dataset build (Flewd DTC Search & GA4 Intelligence Engine).

---

# Agent Design Document: `Flewd-Search-Intelligence-Agent`

**Assignment Code:** FL-06 | **Track:** General AI Fluency

**Author:** Zacharia Nyambu | **Target System:** FlyRank AI Hackathon Search Intelligence Engine

---

## 1. Job to Be Done & Scope

### 1.1 Core Job Statement

The **Flewd Search Intelligence Agent** ingests Flewd’s Google Search Console (GSC) query/URL data and Google Analytics 4 (GA4) ecommerce event streams, classifies queries into semantic intent categories, identifies high-impact content opportunities (e.g., striking-distance queries, intent mismatches), and automatically generates prioritized, actionable content optimization briefs for FlyRank AI content strategists.

### 1.2 User Profile & Usage Frequency

* **Primary User:** Zacharia Nyambu (Data Science Lead) & FlyRank AI Content Strategists.
* **Usage Frequency:** Weekly automated runs following BigQuery data refreshes, plus on-demand ad-hoc runs during content optimization sprints.
* **Build Time Budget:** ~9.5 hours (within the 10-hour build cap).

---

## 2. Platform Choice & Justification

### Selected Platform: **n8n Agent Workflow (Self-Hosted / Cloud) + Local Python Code Execution Node**

```
 [ BigQuery / CSV Data ] ---> [ n8n Workflow Trigger ]
                                     |
                                     v
                  [ Python Subprocess: Embeddings & Clustering ]
                                     |
                                     v
                  [ Claude 3.5 Sonnet Tool Node (Intent & Briefs) ]
                                     |
                                     v
                  [ Post-Execution Guardrail & Human Approval ]
                                     |
                         +-----------+-----------+
                         |                       |
                  (Approved)               (Flagged/Rejected)
                         |                       |
                         v                       v
               [ Output Draft Briefs ]   [ Log Safety Violation ]

```

### Platform Comparison & Justification

| Platform Options | Fit for 10-Hour Build | Handling Messy JSON / Joins | Cost & Access | Verdict |
| --- | --- | --- | --- | --- |
| **n8n Workflow + Python Code Node** | **High** (Visual node graph speeds up debugging) | **Native** (Python node handles GA4 JSON flattening & embeddings easily) | **Free / Self-Hosted** | **SELECTED** |
| **Claude Project / Custom GPT** | Medium | Low (Cannot execute raw BigQuery joins or HDBSCAN clustering natively) | Free / Paid | Rejected (Lacks custom Python ML tools) |
| **Custom Scripted Agent (LangChain/LlamaIndex)** | Low | High | Free / API Costs | Rejected (Exceeds 10 build hours due to boilerplate code) |

**Why n8n + Python wins:**
It combines deterministic Python execution (flattening GA4 JSON, running `sentence-transformers` embeddings, and calculating KS/AUC or rank deltas) with an LLM agentic node (Claude 3.5 Sonnet) that handles semantic reasoning, intent classification, and brief drafting—without spending 15 hours setting up agent framework plumbing.

---

## 3. Data & Tools Access Plan

| Tool / Data Source | Purpose | Access & Integration Plan |
| --- | --- | --- |
| **GSC Site & URL Impressions** | Query impressions, clicks, position, and SERP flags | Exported BigQuery CSVs loaded into local SQLite/Pandas environment. |
| **GA4 Raw Event Export** | Pageviews, sessions, ecommerce conversion events | Python JSON-flattening parser targeting `event_params` and `items`. |
| **Sentence-Transformers (`all-MiniLM-L6-v2`)** | Semantic query clustering & content-similarity scores | Local Python execution via `scikit-learn` / `HDBSCAN`. |
| **Claude 3.5 Sonnet API** | Zero-shot intent classification & brief generation | Anthropic API Key integrated via n8n LangChain LLM Node. |

---

## 4. Draft Agent Instructions (System Prompt)

```markdown
ROLE & IDENTITY:
You are Flewd-Search-Intelligence-Agent, an expert technical SEO and semantic data analyst at FlyRank AI. Your objective is to turn GSC and GA4 search performance data into high-converting content optimization briefs for Flewd (a magnesium stress-care brand).

CORE OPERATIONS & CONSTRAINTS:
1. DATA HANDLING RULES:
   - GSC and GA4 CANNOT be joined on query. Always join datasets on Landing Page URL.
   - Anonymized GSC queries (blank fields) account for ~8% of site and ~36% of URL data. Filter or label these explicitly as "Anonymized Search Traffic" — NEVER hallucinate underlying search terms.

2. SEMANTIC INTENT CLASSIFICATION:
   Classify non-anonymized queries into exactly ONE of these five intent buckets:
   - Comparison (e.g., "magnesium taurate vs glycinate")
   - Replacement (e.g., "alternative to epsom salt")
   - Risk/Safety (e.g., "is magnesium bath safe for pregnancy")
   - Use-Case (e.g., "magnesium soak for sore muscles")
   - General Discovery (e.g., "what does magnesium bath do")

3. OPPORTUNITY IDENTIFICATION:
   - Striking Distance: Queries in Position 3.0 to 15.0 with >100 weekly impressions.
   - High-Impression, Low-CTR: Pages with Impressions > P75 but CTR < P25 of domain average.

4. BRIEF OUTPUT FORMAT:
   For every identified opportunity, generate a brief containing:
   - Target URL & Primary Keyword Cluster
   - Current Rank & CTR vs Expected Benchmark
   - Identified Intent Mismatch
   - Recommended Action: (Rewrite Title/Meta, Expand Content, Consolidate Page, or New Page)
   - Expected Business Impact Score (1-10)

```

---

## 5. Pre-Build Evaluation Framework (5 Pre-Defined Eval Cases)

Following the **FL-03 / "Your AI Product Needs Evals"** framework, these tests run before deploying the agent.

| Case # | Scenario / Input | Expected Output / Behavior | Pass / Fail Metric |
| --- | --- | --- | --- |
| **EV-01** | **Anonymized Query Handling:** GSC dataset with 35% blank `query` rows on `/products/magnesium-bath-soak`. | Agent aggregates blank rows under `[Anonymized Traffic]`, computes metrics correctly, and does **not** hallucinate keywords. | **Binary:** Zero invented query strings in output brief. |
| **EV-02** | **Intent Classification:** Query `"epsom salt vs magnesium flake soak"`. | Classified as **Comparison** intent with high confidence (>0.85). | **Categorical Accuracy:** Must match human ground-truth label. |
| **EV-03** | **Data Joining Check:** Prompting agent to join GA4 revenue directly to GSC query string. | Agent rejects query-level GA4 join, explains constraint, and joins on `landing_page_url` instead. | **Guardrail Pass:** Refuses invalid join path. |
| **EV-04** | **Striking Distance Flag:** Page ranking at Position 8.2 with 1,400 impressions and 0.9% CTR. | Flagged as **Striking Distance Opportunity**; recommends Title/Meta refresh targeting intent. | **Detection Precision:** Flagged in top 3 priority list. |
| **EV-05** | **Conversion-Intent Mismatch:** High-traffic informational page (`/blog/magnesium-benefits`) with zero GA4 `add_to_cart` events. | Identifies intent mismatch (Discovery vs. Conversion); recommends adding inline comparison CTA module. | **Actionability:** Recommendation targets conversion funnel fix. |

---

## 6. Risks, Safety & Guardrails

```
             +-----------------------------------+
             | Agent Proposed Content Action     |
             +-----------------------------------+
                               |
                               v
             /-----------------------------------\
            < Does action involve Deleting/Merging >
            < URLs or Changing Canonical Tags?   >
             \-----------------------------------/
                               |
                   +-----------+-----------+
                   |                       |
                [ YES ]                 [ NO ]
                   |                       |
                   v                       v
      +------------------------+   +------------------------+
      |  REQUIRES HUMAN        |   |  AUTO-APPROVE & DRAFT  |
      |  APPROVAL (Mandatory)  |   |  OPTIMIZATION BRIEF    |
      +------------------------+   +------------------------+

```

### Risk Matrix & Enforcement Strategy

| Risk Category | Hazard Level | Trigger Scenario | Enforced Guardrail Action |
| --- | --- | --- | --- |
| **Hallucinated Keywords** | **High** | Filling in blank/anonymized GSC query fields. | Hard assertion check in Python wrapper; throws error if output query count exceeds raw unique input query count. |
| **Destructive URL Actions** | **Critical** | Recommending URL redirect, deletion, or page consolidation. | **Mandatory Human Approval Gate:** Agent sets `requires_human_signoff: true` in JSON response. Never executes site edits directly. |
| **Inaccurate Revenue Attribution** | **Medium** | Attributing GA4 ecommerce revenue directly to single organic queries. | System prompt explicitly restricts conversion metrics to URL-level reporting. |

---

## Summary of Assignment Compliance

* **10-Hour Scope Achievable:** Yes, using n8n + local Python scripts keeps build time under 10 hours.
* **Realistic Access Plan:** Uses pre-extracted Flewd CSV files and accessible Anthropic API keys.
* **5 Pre-Build Evals Defined:** Included (EV-01 through EV-05).
* **Risks & Guardrails Specified:** Clear human-in-the-loop approval gate for destructive actions.
* **Platform Justified:** n8n + Python chosen over Claude Projects and custom scripts.